In [9]:
import subprocess, sys, os
root = os.path.abspath('')
# macOS: gcc -dynamiclib; Linux: gcc -shared
import platform
flag = '-dynamiclib' if platform.system() == 'Darwin' else '-shared'
ext  = 'dylib'      if platform.system() == 'Darwin' else 'so'
cmd  = ['gcc', '-O2', '-Wall', '-fPIC', flag,
        '-o', f'C/libevoca.{ext}', 'C/evoca.c']
r = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
print(r.stdout or '(no stdout)')
if r.returncode != 0:
    print('STDERR:', r.stderr, file=sys.stderr)
    raise RuntimeError('Build failed')
print('Build OK')

(no stdout)
Build OK


In [10]:
import sys, os
sys.path.insert(0, os.path.abspath(''))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import time
from pprint import pprint

from python.evoca_py import EvoCA, make_gol_lut, LUT_BYTES, lut_bit_index
from python.evoca_py import unpack_lut, available_state_init
from python.display  import run as sdl_run
from python.controls import run_with_controls
from python.evoca_py import import_run
from python.controls import available_probes
from python.evoca_explore import evoca_from_scan, evoca_from_scan_top

In [11]:
N = 512
rng3 = np.random.default_rng(7)
gol_lut=make_gol_lut()
sim = EvoCA()
sim.init(N, food_inc=0.0, m_scale=0.0)
sim.set_lut_all(gol_lut)
sim.set_egenome_all(0)
sim.set_v(rng3.integers(0, 2, (N, N), dtype=np.uint8))
sim.set_f_all(0.0)
sim.set_F_all(0.0)

In [12]:
foo = available_probes()
pprint(foo)

{'activity': 'LUT genome activity (scrolling hash-colored strip)',
 'births': 'Mean +/- std of births array',
 'eg_activity': 'Egenome activity (scrolling hash-colored strip)',
 'eg_food': 'Egene food intake (scrolling hash-colored strip; cumulative food '
            'per egene byte, mouthfuls split across max-match-tied winners)',
 'eg_pop': 'Stacked area: egenome population fractions',
 'egene': 'Egene cognitive stats: 3 sub-strips for mean cognitive specificity '
          '(non-* cell-positions per active egene), mean per-cell cognitive '
          'load, and mean food intake per eater',
 'egenome': 'Egenome stats: mean Negene with +/- std band (top) plus three '
            'sub-strips for distinct egene values, mean max-match, and frac at '
            'Negene_max',
 'entropy': 'Local-pattern Shannon entropy',
 'env_food': 'Mean +/- std of environmental food F(x)',
 'lut_complexity': 'Stacked area: LUT ring-dependency level',
 'n_activity': 'N-activity: Channon shadow LUT-hash s

In [9]:
params = {'N':N,
          'food_inc':0.12,
          'm_scale':0.4,
          'mu_lut':0.001,
          'mu_egene':0.1,
          'tax':0.05,
          'gdiff': 4,
          'restricted_mu':False}
params_state = dict(lut='gol', lut_n_init=1,
                  alive='fraction',
                  alive_fraction=0.5,
                  egenome='uniform',
                    egenome_value=0b000011,
                  f_init=0.1,
                  F_init=0.5)
sim.init(**params)
sim.state(**params_state)

probes = {'ts': True,
          'eg_activity': True,
          'activity': True,
          'egenome': True,
          'egene': True
         }
run_with_controls(sim, probes=probes)

EvoCA: ts probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_144223_ts_manual.csv
EvoCA: egenome probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_144223_egenome_manual.csv
EvoCA: egene probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_144223_egene_manual.csv


<Thread(evoca-sim, started daemon 6437744640)>

EvoCA SDL: starting  N=512 px=2  probes=['--eg-activity=psm_52a753b4']
EvoCA SDL: probe SharedMemory open failed: [Errno 2] No such file or directory: '/--activity=psm_4f2eeff2'
EvoCA SDL: activity shm opened (256x512)
EvoCA SDL: eg_activity shm opened (256x512)
EvoCA SDL: egenome shm opened (5x512)
EvoCA SDL: egene shm opened (3x512)
EvoCA SDL: ts shm opened (6 traces)


In [5]:
# Suggested params for seeing egene/egenome evolution.
# Slow mutation lets selection stabilise cognition; mu_egenome>0
# turns on Negene growth; small tax_per_egene gives gentle
# per-position pressure (option-b tax). Watch the egenome
# probe's `Ngene` and the egene probe's `spec`/`load`/`food`
# strips drift over a few thousand ticks.
params = dict(
    N=N,
    food_inc=0.013,
    m_scale=1.2,
    gdiff=0.06,
    mu_lut=0.001,
    mu_egene=0.003,        # 30x slower than the previous 0.1
    mu_egenome=0.005,      # let Negene drift up
    p_dup_egene=1.0,       # default — new slot duplicates a random active one
    tax=0.035,
    tax_per_egene=0.0001,  # gentle per-position pressure (option b)
    tax_lut=0.0,
    restricted_mu=True,
)
params_state = dict(
    lut='gol', lut_n_init=1,
    alive='halfplane',
    egenome='uniform', egenome_value=0b000011,
    f_init=0.5,
    F_init=1.0,
)
sim.init(**params)
sim.state(**params_state)

probes = {
    'ts':       True,
    'eg_activity': True,
    'eg_food':  True,
    'activity': True,
    'egenome':  True,
    'egene':    True,
}
run_with_controls(sim, probes=probes)


EvoCA: ts probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_154619_ts_manual.csv
EvoCA: egenome probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_154619_egenome_manual.csv
EvoCA: egene probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_154619_egene_manual.csv


<Thread(evoca-sim, started daemon 6287503360)>

In [14]:
# Watch cognition grow from a low starting point.
# Seed with a minimum-survivable specification (centre + axis-1 = 2
# orbits, so spec starts at 5 cell-positions), low tax, and gentle
# per-position pressure. With mu_egenome > 0 the cell can also add
# more egenes; with low mu_egene the existing mask drifts slowly so
# selection has time to act.
params = dict(
    N=N,
    food_inc=0.02,
    m_scale=1.5,
    gdiff=0.05,
    mu_lut=0.0005,
    mu_egene=0.001,
    mu_egenome=0.003,
    p_dup_egene=1.0,
    tax=0.020,
    tax_per_egene=0.00005,   # very gentle: max ≈ 0.01 / tick / cell
    tax_lut=0.0,
    restricted_mu=True,
)
sim.init(**params)
sim.state(lut='gol', lut_n_init=1, alive='halfplane',
          f_init=0.6, F_init=1.0)

# Override the default egenome init (mask=0x3F, fully specified) with
# a small starting cognition: value=0, mask=0b000011 (centre + axis-1
# orbits both expecting 0). spec starts at 1 + 4 = 5.
sim.set_egenome_pair_all(value=0b000000, mask=0b000011)

print('Initial cell (N//2, N//2):', sim.cell_inspect(N//2, N//2))
print('Initial egene_stats:', sim.egene_stats())

probes = {
    'ts':       True,
    'eg_activity': True,
    'eg_food':  True,
    'activity': True,
    'egenome':  True,
    'egene':    True,
}
run_with_controls(sim, probes=probes)


Initial cell (N//2, N//2): {'alive': False, 'active_byte': 1, 'Negene': 1, 'slots': [{'slot': 0, 'value': 0, 'mask': 3, 'orbits_b0..b5': '00....'}]}
Initial egene_stats: {'mean_specificity': 5.0, 'mean_load': 5.0, 'mean_intake': 0.0}


EvoCA: ts probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_185125_ts_manual.csv
EvoCA: egenome probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_185125_egenome_manual.csv
EvoCA: egene probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_185125_egene_manual.csv


<Thread(evoca-sim, started daemon 12935360512)>

In [ ]:
# Scan-3 productive corner — the parameter region that produced
# the reaction-diffusion waves and cyclic dying-off-and-reblooming
# in the previous scans (Scans/2026-04-27_*). Higher tax + lower
# food than the cog-grow-from-low cell, so selection pressure is
# back. The new ternary cognition machinery is set to the
# backward-compat regime (mu_egenome=0, tax_per_egene=0, default
# mask=0x3F) — every cell starts fully specified, only mu_egene
# drives drift away from that. Should restore the R-D dynamics.
params = dict(
    N=N,
    food_inc=0.013,
    m_scale=1.2,
    gdiff=0.06,
    mu_lut=0.001,
    mu_egene=0.003,
    mu_egenome=0.0,        # off — Negene stays at 1
    p_dup_egene=1.0,
    tax=0.035,
    tax_per_egene=0.0,
    tax_lut=0.0,
    restricted_mu=True,
)
sim.init(**params)
sim.state(lut='gol', lut_n_init=1, alive='halfplane',
          egenome='uniform', egenome_value=0b000011,
          f_init=0.5, F_init=1.0)

probes = {
    'ts':       True,
    'eg_activity': True,
    'eg_food':  True,
    'activity': True,
    'egenome':  True,
    'egene':    True,
}
run_with_controls(sim, probes=probes)


## Detecting evolutionary progress

Two probes look like they should both answer "are agents getting better at eating?", but they measure different layers:

- **`egenome:match`** is the raw signed score from `fiducial_matches_best`, *before* the headroom clamp and *before* multiplying by `F_food`. Range `[-25, +25]`.
- **`egene:food`** is the actual mouthful eaten this step: `(m_scale/25) × max(0, match) × F_food × clamp(headroom)`. Range `[0, 1]`.

The food field is a bottleneck: `food_inc` pumps food in at a fixed rate; the population eats it out. **In steady state, average alive cell food intake converges to `tax`** — anything less and cells die; anything more and pop grows until the per-cell share drops to tax. So `egene:food` flat-lining at roughly `tax` is *carrying-capacity equilibrium*, not a failure of evolution.

`egenome:match` can keep rising at equilibrium because evolution can improve the matching pattern *within* the fixed food budget. As agents match better, the food field gets depleted further; the raw match score rises while the resulting mouthful stays pinned at tax. The Red Queen race: the same bite of food extracted by a sharper tooth.

**Evolutionary signals at carrying-capacity equilibrium:**

1. **`egenome:match` rising** — cognition improving.
2. **`ts:pop` rising slowly** at stable food/cell — the same food supports a denser population.
3. **Channon excess activity slope `> 0`** (via `ts:activity_flux`) — selection beating random.

**Clean test of "evolution vs. drift":** run identical params with all mutation rates `= 0` (the `frozen-baseline` cell below). Compare steady-state `match`. If evolved-match > frozen-match by more than per-run noise, evolution is doing real work.

---

## Fixed LUT / evolving egenomes

The cells below the baseline experiment with **`mu_lut = 0`** plus a single uniform LUT, so spatial dynamics are *frozen* and the same across the population. The only thing evolution can adjust is the egenome — pure cognition-against-fixed-substrate. Two substrates are offered: a random `n_init` ring-restricted LUT (parameter sweep over density) and Conway's GoL (`make_gol_lut()`). GoL is well known to settle quickly to fixed points + small oscillators on a finite torus; for sustained dynamics, a random `n_init=2` LUT with density around 0.4 is usually more "alive".


In [ ]:
# Frozen baseline — same params as scan3-productive-corner above
# but every mutation rate is 0. Cells inherit their parents' LUT and
# egenome verbatim, no drift. The only thing that varies across
# the population is which cell happens to be where in the lattice
# and the dynamical CA state.
#
# Use this to test whether the slow rise in egenome:match you see
# in the evolving version is evolution doing real work, or just
# drift settling. Run this cell first to a steady-state match,
# then run the evolving cell — if evolved-match > frozen-match by
# more than the per-run noise, evolution is paying off.
params = dict(
    N=N,
    food_inc=0.013,
    m_scale=1.2,
    gdiff=0.06,
    mu_lut=0.0,            # frozen
    mu_egene=0.0,          # frozen
    mu_egenome=0.0,        # frozen
    p_dup_egene=1.0,
    tax=0.035,
    tax_per_egene=0.0,
    tax_lut=0.0,
    restricted_mu=True,
)
sim.init(**params)
sim.state(lut='gol', lut_n_init=1, alive='halfplane',
          egenome='uniform', egenome_value=0b000011,
          f_init=0.5, F_init=1.0)

probes = {
    'ts':       True,
    'eg_activity': True,
    'eg_food':  True,
    'activity': True,
    'egenome':  True,
    'egene':    True,
}
run_with_controls(sim, probes=probes)


In [ ]:
# Fixed LUT / evolving egenome: a single uniform LUT shared by every
# cell, mu_lut = 0 so spatial dynamics are frozen. Egenes are still
# free to mutate (mu_egene, mu_egenome > 0) so cognition can adapt to
# this fixed substrate.
#
# Try a few LUT recipes and watch egene:spec, egenome:match, and the
# eg_activity strip evolve.
params = dict(
    N=N,
    food_inc=0.013,
    m_scale=1.2,
    gdiff=0.06,
    mu_lut=0.0,                 # frozen LUT
    mu_egene=0.003,             # egene bits drift / are selected
    mu_egenome=0.005,           # Negene can grow
    p_dup_egene=1.0,
    tax=0.035,
    tax_per_egene=0.0001,
    tax_lut=0.0,
    restricted_mu=True,
)
sim.init(**params)
sim.state(lut='gol',            # placeholder; overridden below
          alive='halfplane',
          egenome='uniform', egenome_value=0b000011,
          f_init=0.5, F_init=1.0)

# Pick one substrate by uncommenting:

# (a) Random LUT, conditioning on n1+n2 only, modest density.
#     Reproducible via the seed; try density in [0.3, 0.6] for variety.
from python.evoca_py import make_random_lut, make_gol_lut
import ctypes
lut = make_random_lut(n_init=2, density=0.4, seed=7)
sim._lib.evoca_set_lut_all(lut.ctypes.data_as(ctypes.POINTER(ctypes.c_uint8)))

# (b) GoL — settles quickly; comment (a) and uncomment to compare.
# sim._lib.evoca_set_lut_all(make_gol_lut().ctypes.data_as(
#     ctypes.POINTER(ctypes.c_uint8)))

probes = {
    'ts':       True,
    'eg_activity': True,
    'eg_food':  True,
    'activity': True,
    'egenome':  True,
    'egene':    True,
}
run_with_controls(sim, probes=probes)


In [9]:
sim.egenome_stats()

{'mean_negene': 2.129810333251953,
 'std_negene': 1.0412410497665405,
 'distinct_egene_values': 63,
 'mean_max_match': 18.293813705444336,
 'frac_at_max': 0.0}

In [11]:
foo = sim.get_egenome()

In [14]:
foo=sim.get_alive()

In [20]:
fooo = foo.flatten()
np.sum(fooo)/len(fooo)

np.float64(0.21223068237304688)

In [21]:
foo=sim.get_egenes_mask()
foo[0][:10]

array([[61, 29, 63, 47, 57,  5, 30, 60],
       [61, 29, 63, 47, 57,  5, 30, 60],
       [62, 63, 60, 63, 13, 61, 47, 61],
       [62, 63, 60, 63, 13, 63, 47, 61],
       [62, 63, 60, 63, 13, 63, 47, 61],
       [62, 63, 60, 63, 13, 63, 47, 61],
       [62, 63, 28, 21,  7,  7, 63,  7],
       [62, 63, 28, 21,  7,  7, 63,  7],
       [62, 63, 28, 21,  7,  7, 63,  7],
       [62, 63, 28, 21,  7,  7, 63,  7]], dtype=uint8)

In [23]:
foo = sim.get_egenome()
foo[0][:10]

array([3, 3, 1, 1, 1, 1, 1, 1, 1, 1], dtype=uint8)

In [ ]:
sim.get_eg